# Limpieza, análisis y estructuración de datos con Pandas (Instacart)

**Actividad:** GA1-220501094-01-AA1-EV06

**Pregunta orientadora:** ¿Cómo preparar y estructurar los datos de Instacart para obtener información confiable sobre los hábitos de compra?

Este cuaderno contiene los laboratorios guiados y los retos de las lecciones 2, 3 y 4. Los ejemplos de la explicación están en `pandas_examples.ipynb`.

## Preparación

Carga de los datos crudos y limpieza de la explicación de la Lección 2. Las tablas `orders`, `products` y `order_prods` limpias se usan en los laboratorios y retos siguientes.

In [32]:
import pandas as pd
import numpy as np
from pathlib import Path

RUTA = Path('../data/raw')
SALIDA = Path('../data/processed')

orders      = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
products    = pd.read_csv(RUTA / 'products.csv', sep=';')
order_prods = pd.read_csv(RUTA / 'order_products.csv', sep=';')
aisles      = pd.read_csv(RUTA / 'aisles.csv', sep=';')
departments = pd.read_csv(RUTA / 'departments.csv', sep=';')

print(pd.__version__)
print(orders.shape, products.shape, order_prods.shape, aisles.shape, departments.shape)
# Esperado: (478967, 6) (49694, 4) (4545007, 4) (134, 2) (21, 2)

3.0.6
(478967, 6) (49694, 4) (4545007, 4) (134, 2) (21, 2)


In [33]:
# Limpieza manual de la explicación de la Lección 2
orders = orders.drop_duplicates().reset_index(drop=True)
orders = orders.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                        'order_dow': 'int8', 'order_hour_of_day': 'int8',
                        'days_since_prior_order': 'Int16'})

order_prods = order_prods.astype({'order_id': 'int32', 'product_id': 'int32',
                                  'add_to_cart_order': 'Int16', 'reordered': 'int8'})

products['product_name'] = (products['product_name'].fillna('Unknown')
                            .str.replace('\xa0', ' ', regex=False)
                            .str.replace(r'\s+', ' ', regex=True)
                            .str.strip().str.title())

assert orders.shape == (478952, 6)
assert orders['days_since_prior_order'].isna().sum() == 28817
assert products['product_name'].isna().sum() == 0
assert str(order_prods['add_to_cart_order'].dtype) == 'Int16'
print('Tablas limpias listas')

Tablas limpias listas


# Lección 2: Limpieza y estandarización

## Laboratorio guiado 2: pipeline de limpieza con informe de cambios

Objetivo: construir una secuencia de pasos de limpieza que parte de los archivos crudos, se ejecuta con una sola instrucción y deja un informe de lo que cambió en cada paso.

### Paso 1: recargar los datos crudos

In [34]:
orders_crudo   = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')
op_crudo       = pd.read_csv(RUTA / 'order_products.csv', sep=';')
print(orders_crudo.shape, products_crudo.shape, op_crudo.shape)
# Esperado: (478967, 6) (49694, 4) (4545007, 4)

(478967, 6) (49694, 4) (4545007, 4)


### Paso 2: una función que anota el estado de la tabla

In [35]:
registro = []

def anotar(df, tabla, paso):
    registro.append({
        'tabla': tabla,
        'paso': paso,
        'filas': len(df),
        'nulos': int(df.isna().sum().sum()),
        'memoria_mb': round(df.memory_usage(deep=True).sum() / 1e6, 1),
    })
    return df

### Paso 3: un paso de limpieza por función

In [36]:
def quitar_duplicados(df):
    return df.drop_duplicates().reset_index(drop=True)

def tipos_orders(df):
    return df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                    'order_dow': 'int8', 'order_hour_of_day': 'int8',
                    'days_since_prior_order': 'Int16'})

def limpiar_nombres(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                        .str.replace('\xa0', ' ', regex=False)
                        .str.replace(r'\s+', ' ', regex=True)
                        .str.strip().str.title())
    return df

def tipos_order_prods(df):
    return df.astype({'order_id': 'int32', 'product_id': 'int32',
                    'add_to_cart_order': 'Int16', 'reordered': 'int8'})

### Paso 4: encadenar los pasos con `pipe`

Si `registro` tiene filas repetidas, vuelve a ejecutar desde la celda `registro = []`.

In [37]:
orders_limpio = (orders_crudo
    .pipe(anotar, 'orders', '0 crudo')
    .pipe(quitar_duplicados)
    .pipe(anotar, 'orders', '1 sin duplicados')
    .pipe(tipos_orders)
    .pipe(anotar, 'orders', '2 tipos'))

products_limpio = (products_crudo
    .pipe(anotar, 'products', '0 crudo')
    .pipe(limpiar_nombres)
    .pipe(anotar, 'products', '1 nombres'))

op_limpio = (op_crudo
    .pipe(anotar, 'order_prods', '0 crudo')
    .pipe(tipos_order_prods)
    .pipe(anotar, 'order_prods', '1 tipos'))

### Paso 5: el informe de cambios

In [38]:
informe = pd.DataFrame(registro)
informe.to_csv(SALIDA / 'cleaning_report.csv', index=False)
informe

,tabla,paso,filas,nulos,memoria_mb
0,orders,0 crudo,478967,28819,23.0
1,orders,1 sin duplicados,478952,28817,23.0
2,orders,2 tipos,478952,28817,7.2
3,products,0 crudo,49694,1258,3.1
4,products,1 nombres,49694,0,3.1
5,order_prods,0 crudo,4545007,836,145.4
6,order_prods,1 tipos,4545007,836,54.5


**Interpretación del informe:**

- Quitar duplicados eliminó 15 filas de `orders` y 2 nulos: dos de los pedidos repetidos eran primeros pedidos.
- Corregir los tipos no cambió filas ni nulos, pero redujo la memoria de `orders` de 23,0 a 7,2 MB y la de `order_prods` de 145,4 a 54,5 MB.
- Los 1 258 nulos de `product_name` se rellenaron con `'Unknown'`.
- Los 836 nulos de `order_prods` se conservan a propósito (ver Reto 2, punto 3).

El informe se guarda en `data/processed/cleaning_report.csv`.

### Paso 6: validar el resultado

In [39]:
def validar(o, p, op):
    assert o['order_id'].is_unique, 'order_id repetido'
    assert o['order_hour_of_day'].between(0, 23).all(), 'hora fuera de rango'
    assert p['product_name'].notna().all(), 'productos sin nombre'
    assert not p['product_name'].str.contains('\xa0', regex=False).any(), 'espacios invisibles'
    assert op['reordered'].isin([0, 1]).all(), 'reordered con valores raros'
    return 'Validación superada'

print(validar(orders_limpio, products_limpio, op_limpio))
print(orders_limpio.equals(orders), products_limpio.equals(products))
# Esperado: Validación superada / True True

Validación superada
True True


### Conclusiones del laboratorio

- El pipeline produce exactamente las mismas tablas que la limpieza manual, pero se puede volver a ejecutar con una sola instrucción cuando lleguen datos nuevos.
- El informe responde con cifras a la pregunta "¿qué le hiciste a los datos?".
- La función `validar` convierte las reglas de calidad en una prueba que se ejecuta cada vez.

## Reto 2

### 1. Duplicados y valores de `order_prods`

¿Hay filas duplicadas exactas? ¿Se repite algún par `order_id` + `product_id`? ¿`reordered` solo contiene 0 y 1?

In [40]:
print('Filas duplicadas exactas:', order_prods.duplicated().sum())
print('Pares order_id + product_id repetidos:',
      order_prods.duplicated(subset=['order_id', 'product_id']).sum())
print('Valores de reordered:', sorted(order_prods['reordered'].unique().tolist()))
print('¿Solo 0 y 1?', order_prods['reordered'].isin([0, 1]).all())

Filas duplicadas exactas: 0
Pares order_id + product_id repetidos: 0
Valores de reordered: [0, 1]
¿Solo 0 y 1? True


**Interpretación:** `order_prods` no tiene filas duplicadas exactas ni pares `order_id` + `product_id` repetidos, así que esa combinación funciona como clave compuesta: cada producto aparece una sola vez por pedido. `reordered` solo contiene 0 y 1. Esta tabla no necesita limpieza de duplicados ni de valores.

### 2. Texto en `aisles` y `departments`

¿Tienen nombres duplicados o con espacios al inicio o al final? ¿Necesitan limpieza de texto?

In [41]:
for nombre, df, col in [('aisles', aisles, 'aisle'),
                       ('departments', departments, 'department')]:
    texto = df[col]
    print(nombre)
    print('  nulos:', texto.isna().sum())
    print('  duplicados exactos:', texto.duplicated().sum())
    print('  duplicados al ignorar mayúsculas y espacios:',
          texto.str.lower().str.strip().duplicated().sum())
    print('  con espacios al inicio o al final:', (texto != texto.str.strip()).sum())
    print('  con espacios dobles:', texto.str.contains('  ', regex=False).sum())

aisles
  nulos: 0
  duplicados exactos: 0
  duplicados al ignorar mayúsculas y espacios: 0
  con espacios al inicio o al final: 0
  con espacios dobles: 0
departments
  nulos: 0
  duplicados exactos: 0
  duplicados al ignorar mayúsculas y espacios: 0
  con espacios al inicio o al final: 0
  con espacios dobles: 0


**Interpretación:** `aisles` y `departments` no tienen nulos, nombres duplicados (ni siquiera al ignorar mayúsculas y espacios) ni espacios sobrantes. No necesitan limpieza de texto. Lo único a tener en cuenta es el valor centinela `missing` de `departments` (punto 7).

### 3. Los 836 nulos de `add_to_cart_order`

¿En cuántos pedidos ocurren? ¿Cuántos productos tiene, como mínimo, cada uno de esos pedidos? ¿Qué los explica?

In [42]:
nulos_carrito = order_prods[order_prods['add_to_cart_order'].isna()]
pedidos_con_nulos = nulos_carrito['order_id'].unique()
print('Filas con nulo:', len(nulos_carrito))
print('Pedidos afectados:', len(pedidos_con_nulos))

productos_por_pedido = order_prods.groupby('order_id').size()
print('Mínimo de productos en esos pedidos:', productos_por_pedido.loc[pedidos_con_nulos].min())
print('Posición máxima registrada en add_to_cart_order:', order_prods['add_to_cart_order'].max())

Filas con nulo: 836
Pedidos afectados: 70
Mínimo de productos en esos pedidos: 65
Posición máxima registrada en add_to_cart_order: 64


**Interpretación:** los 836 nulos están en solo 70 pedidos, y cada uno de esos pedidos tiene al menos 65 productos. Como la posición máxima registrada en `add_to_cart_order` es 64, el sistema solo guarda la posición del carrito hasta el producto 64: a partir del 65 queda vacía. No es un error de los datos sino un límite del registro, por eso se **conservan** los nulos (tipo `Int16`). Rellenarlos inventaría posiciones.

### 4. Nombres que cambiaron con la estandarización

Cuenta cuántos nombres (no nulos) de `products.csv` cambiaron con la limpieza.

In [43]:
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')

crudos = products_crudo['product_name']
limpios = products['product_name']
no_nulos = crudos.notna()

cambiaron = (crudos[no_nulos] != limpios[no_nulos])
print('Nombres no nulos:', no_nulos.sum())
print('Nombres que cambiaron:', cambiaron.sum())
pd.DataFrame({'antes': crudos[no_nulos][cambiaron], 'despues': limpios[no_nulos][cambiaron]}).head()

Nombres no nulos: 48436
Nombres que cambiaron: 6321


,antes,despues
17,Pizza for One Suprema Frozen Pizza,Pizza For One Suprema Frozen Pizza
23,Tri-Vi-Sol® Vitamins A-C-and D Supplement Drop...,Tri-Vi-Sol® Vitamins A-C-And D Supplement Drop...
29,Three Cheese Ziti- Marinara with Meatballs,Three Cheese Ziti- Marinara With Meatballs
35,Traditional Lasagna with Meat Sauce Savory Ita...,Traditional Lasagna With Meat Sauce Savory Ita...
41,Biotin 1000 mcg,Biotin 1000 Mcg


**Interpretación:** de 48 436 nombres no nulos, 6 321 cambiaron con la estandarización (cerca del 13 %). La mayoría de los cambios son de mayúsculas por `.str.title()` (por ejemplo, `for` → `For`) y una parte son espacios dobles o invisibles. Una limitación: `.title()` también cambia siglas y unidades (`mcg` → `Mcg`), lo cual no afecta el análisis porque se agrupa por `product_id`.

### 5. Día de la semana como categoría ordenada

Series con el nombre del día de cada pedido, convertida a categoría ordenada de domingo a sábado. Comparación de memoria.

In [44]:
dias = ['domingo', 'lunes', 'martes', 'miércoles', 'jueves', 'viernes', 'sábado']

nombre_dia = orders['order_dow'].map(dict(enumerate(dias)))
dia_categoria = pd.Series(pd.Categorical(nombre_dia, categories=dias, ordered=True),
                          index=nombre_dia.index)

mem_texto = nombre_dia.memory_usage(deep=True) / 1e6
mem_categoria = dia_categoria.memory_usage(deep=True) / 1e6
print(f'Texto: {mem_texto:.2f} MB')
print(f'Categoría: {mem_categoria:.2f} MB')
print(f'La categoría ocupa {mem_categoria / mem_texto:.1%} del texto')
print(dia_categoria.value_counts().sort_index())

Texto: 7.14 MB
Categoría: 0.48 MB
La categoría ocupa 6.7% del texto
domingo      84090
lunes        82185
martes       65833
miércoles    60897
jueves       59810
viernes      63488
sábado       62649
Name: count, dtype: int64


**Interpretación:** el texto ocupa 7,14 MB y la categoría 0,48 MB, apenas el 6,7 % (unas 15 veces menos), porque la categoría guarda cada uno de los 7 nombres una sola vez y en cada fila solo un código. Al ser **ordenada**, `sort_index()` muestra los días de domingo a sábado y no en orden alfabético. El domingo es el día con más pedidos (84 090) y el jueves el de menos (59 810).

### 6. Productos `'Unknown'` en los pedidos

¿Cuántas filas de `order_prods` corresponden a productos `'Unknown'` y en cuántos pedidos aparecen? ¿Los eliminarías?

In [45]:
ids_unknown = products.loc[products['product_name'] == 'Unknown', 'product_id']
filas_unknown = order_prods[order_prods['product_id'].isin(ids_unknown)]

print('Filas con productos Unknown:', len(filas_unknown))
print('Pedidos en los que aparecen:', filas_unknown['order_id'].nunique())
print(f'Porcentaje de las filas de order_prods: {len(filas_unknown) / len(order_prods):.2%}')

Filas con productos Unknown: 9547
Pedidos en los que aparecen: 8254
Porcentaje de las filas de order_prods: 0.21%


**Interpretación:** hay 9 547 filas de `order_prods` con productos `Unknown`, repartidas en 8 254 pedidos. Son solo el 0,21 % de las filas, pero son compras reales.

**¿Los eliminarías?** No. Son compras reales: el producto existe y tiene `product_id`, aunque falte su nombre en el catálogo. Eliminarlos reduciría el tamaño real de esos pedidos y alteraría métricas como la tasa de recompra o los artículos por pedido. Es mejor conservarlos y, si un análisis lo necesita, excluirlos solo en ese análisis.

### 7. `'missing'` como nulo real en una copia de `departments`

In [46]:
deptos = departments.copy()
deptos['department'] = deptos['department'].replace('missing', pd.NA)

print('Nulos en la copia:', deptos['department'].isna().sum())
print('Nulos en la original:', departments['department'].isna().sum())
deptos[deptos['department'].isna()]

Nulos en la copia: 1
Nulos en la original: 0


,department_id,department
20,21,NaN


**¿Por qué no hacerlo en la tabla original antes de agrupar?** Porque `groupby` descarta por defecto las claves nulas (`dropna=True`): los productos de ese departamento desaparecerían en silencio de los resúmenes, mientras que con el texto `'missing'` siguen visibles como un grupo más.